In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [22]:

# Dummy data (context-response pairs)

data = [
    ("hi", "hello"),
    ("how are you", "i am fine"),
    ("what are you doing", "just working"),
    ("good morning", "morning!"),
    ("bye", "see you"),
    ("what is your hobby", "i like play tennis"),
]


In [23]:
# Add special tokens
special_tokens = ["<PAD>", "<SOS>", "<EOS>"]
vocab = list(set(" ".join([f"{c} {r}" for c, r in data]).replace("!", "").split()))
vocab = special_tokens + vocab
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)
embed_dim, hidden_dim, latent_dim = 32, 64, 16


In [ ]:

# Encoder

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        emb = self.embedding(x)
        # Assuming single layer GRU
        _, h = self.rnn(emb)
        mu = self.mu(h.squeeze(0))
        logvar = self.logvar(h.squeeze(0))
        return mu, logvar



In [26]:

# Decoder

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, z):
        emb = self.embedding(x)
        h0 = self.latent_to_hidden(z).unsqueeze(0)
        output, _ = self.rnn(emb, h0)
        logits = self.out(output)
        return logits

In [27]:

# CVAE Model (Updated with BOW MLP)

class CVAE(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_dim, latent_dim)
        self.vocab_size = vocab_size

        self.mlp_bow = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size)
        ).to(device)


    def forward(self, x, y):
        mu, logvar = self.encoder(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std

        if y.size(1) < 2:
            # Handle minimal response length for decoder input
            return None, mu, logvar, None

        logits = self.decoder(y[:, :-1], z)

        # BOW Prediction
        bow_logits = self.mlp_bow(z)

        return logits, mu, logvar, bow_logits

    @torch.no_grad()
    def generate_diversity(self, x_context, num_samples=3, max_len=8):
        """Generates multiple diverse responses by sampling 'z' multiple times."""
        self.eval()

        # Calculate mu, logvar once from the context
        mu, logvar = self.encoder(x_context.to(device))

        responses = []
        for _ in range(num_samples):
            # 1. Sample latent variable z from the posterior
            std = torch.exp(0.5 * logvar)
            z = mu + torch.randn_like(mu) * std

            # 2. Sequential Decoding (Conditioned on z)
            inp = torch.tensor([[word2idx["<SOS>"]]], dtype=torch.long).to(device)
            outputs = []

            for _ in range(max_len):
                logits = self.decoder(inp, z)
                probs = F.softmax(logits[:, -1, :], dim=-1)
                # Sample the next token (instead of greedy argmax, for more natural diversity)
                next_token = torch.multinomial(probs, num_samples=1)
                next_token_id = next_token.item()

                if next_token_id == word2idx["<EOS>"]:
                    break

                # Exclude <PAD> token from output
                if next_token_id == word2idx["<PAD>"]:
                    break

                outputs.append(next_token_id)
                inp = torch.cat([inp, next_token], dim=1)

            responses.append(" ".join([idx2word[i] for i in outputs]))

        self.train()
        return responses


In [28]:
# --------------------------
# Utilities
# --------------------------
def sentence_to_tensor(sentence, add_sos=False):
    # Use <PAD> for OOV words
    tokens = [word2idx.get(w, word2idx["<PAD>"]) for w in sentence.split()]
    if add_sos:
        tokens = [word2idx["<SOS>"]] + tokens
    tokens.append(word2idx["<EOS>"])
    return torch.tensor([tokens], dtype=torch.long).to(device)

def get_bow_target(y_tensor, vocab_size):
    """Generates the Binary Bag-of-Words target vector from the response tensor."""
    # y_tensor is (1, seq_len)
    bow_target = torch.zeros(y_tensor.size(0), vocab_size).to(device)

    # Get all unique words in the response (excluding special tokens)
    special_indices = [word2idx[t] for t in ["<SOS>", "<EOS>", "<PAD>"]]
    unique_words = [idx for idx in y_tensor.flatten().unique().tolist() if idx not in special_indices]

    # Set the corresponding indices in the target vector to 1.0
    if unique_words:
        bow_target[0].scatter_(0, torch.tensor(unique_words).to(device), 1.0)

    return bow_target

In [29]:

# Training

print(f"Using device: {device}")
model = CVAE(vocab_size, embed_dim, hidden_dim, latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 500

# Loss Weights (as suggested by the paper, and common practice)
KL_WEIGHT = 0.01
BOW_WEIGHT = 0.1

print(f"Starting training on {len(data)} pairs for {epochs} epochs...")

for epoch in range(epochs):
    total_loss, total_recon_loss, total_kl_loss, total_bow_loss = 0, 0, 0, 0

    for context, response in data:
        x = sentence_to_tensor(context)
        y = sentence_to_tensor(response, add_sos=True)

        # Updated model call returns 4 results
        result = model(x, y)
        if result[0] is None: continue # Skip if response is too short

        logits, mu, logvar, bow_logits = result

        # 1. Reconstruction Loss (Cross-Entropy)
        recon_loss = F.cross_entropy(
            logits.reshape(-1, vocab_size), y[:, 1:].reshape(-1)
        )

        # 2. KL Divergence Loss
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

        # 3. Bag-of-Word (BOW) Loss (Uses the BOW MLP)
        bow_target = get_bow_target(y, vocab_size)
        # Binary Cross-Entropy with Logits is suitable for multi-label classification (word presence)
        bow_loss = F.binary_cross_entropy_with_logits(bow_logits, bow_target)

        # Total Loss (L') = Recon + KL * KL_Weight + BOW * BOW_Weight
        loss = recon_loss + KL_WEIGHT * kl_loss + BOW_WEIGHT * bow_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon_loss += recon_loss.item()
        total_kl_loss += kl_loss.item() * KL_WEIGHT
        total_bow_loss += bow_loss.item() * BOW_WEIGHT

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:03d} | Total Loss: {total_loss:.4f} | Recon: {total_recon_loss:.4f} | KL: {total_kl_loss:.4f} | BOW: {total_bow_loss:.4f}")


Using device: cuda
Starting training on 6 pairs for 500 epochs...
Epoch 050 | Total Loss: 1.8829 | Recon: 0.8785 | KL: 0.9147 | BOW: 0.0896
Epoch 100 | Total Loss: 1.0148 | Recon: 0.3088 | KL: 0.5997 | BOW: 0.1063
Epoch 150 | Total Loss: 0.6533 | Recon: 0.1422 | KL: 0.4810 | BOW: 0.0301
Epoch 200 | Total Loss: 0.5771 | Recon: 0.0608 | KL: 0.4729 | BOW: 0.0433
Epoch 250 | Total Loss: 0.5256 | Recon: 0.0714 | KL: 0.4285 | BOW: 0.0257
Epoch 300 | Total Loss: 0.7401 | Recon: 0.3783 | KL: 0.3357 | BOW: 0.0261
Epoch 350 | Total Loss: 0.4502 | Recon: 0.1287 | KL: 0.3036 | BOW: 0.0178
Epoch 400 | Total Loss: 0.3731 | Recon: 0.0112 | KL: 0.3561 | BOW: 0.0058
Epoch 450 | Total Loss: 1.6427 | Recon: 1.2852 | KL: 0.3161 | BOW: 0.0414
Epoch 500 | Total Loss: 0.4579 | Recon: 0.0699 | KL: 0.3770 | BOW: 0.0111


In [30]:

# Generate Diverse Responses (Demonstrates CVAE's core function)

print("\n" + "="*50)
test_context = "what are you doing"
x_test = sentence_to_tensor(test_context)
num_responses = 3

print(f"Input Context: '{test_context}'")
print(f"Generating {num_responses} diverse responses by sampling Z:")

diverse_responses = model.generate_diversity(x_test, num_samples=num_responses)

for i, resp in enumerate(diverse_responses):
    print(f"Response {i+1}: {resp}")

print("\n" + "="*50)
test_context = "what is your hobby"
x_test = sentence_to_tensor(test_context)

print(f"Input Context: '{test_context}'")
print(f"Generating {num_responses} diverse responses by sampling Z:")

diverse_responses = model.generate_diversity(x_test, num_samples=num_responses)

for i, resp in enumerate(diverse_responses):
    print(f"Response {i+1}: {resp}")
print("="*50)


Input Context: 'what are you doing'
Generating 3 diverse responses by sampling Z:
Response 1: just working
Response 2: just working
Response 3: just working

Input Context: 'what is your hobby'
Generating 3 diverse responses by sampling Z:
Response 1: i like play tennis
Response 2: i like play tennis
Response 3: i like play tennis
